### SCD tyoe 1 implementation on cleaned customers data

In [0]:
create table if not exists identifier(:catalog || '.silver.customers_scd')
as select * except(ingestion_ts, file_name, file_path)
from identifier(:catalog || '.silver.silver_customers')
where 1=0

In [0]:
merge into identifier(:catalog || '.silver.customers_scd') t
using (
    select customer_id, first_name, last_name, date_of_birth, email, phone, address, city, state, postal_code, customer_segment, customer_status, registration_date, updated_at 
    from identifier(:catalog || '.silver.silver_customers')
    qualify row_number() over (partition by customer_id order by updated_at desc) = 1
) s
on t.customer_id = s.customer_id
when matched then 
update set t.email = s.email,
t.phone = s.phone,
t.address = s.address,
t.city = s.city,
t.state = s.state,
t.postal_code = s.postal_code,
t.updated_at = s.updated_at

when not matched then
insert (customer_id, first_name, last_name, date_of_birth, email, phone, address, city, state, postal_code, customer_segment, customer_status, registration_date, updated_at) values (s.customer_id, s.first_name, s.last_name, s.date_of_birth, s.email, s.phone, s.address, s.city, state, s.postal_code, s.customer_segment, s.customer_status, s.registration_date, s.updated_at)

